**Making Tables (Milestone 2) | Tyler Hilbert | April 14, 2026**

After the brainstorming session in class, I decided the next milestone I am going for is to make tables that both summarize the MT grade data in each class and the intervention tables. These tables each serve different audiences:
- Leadership will reference the summary tables - this shows them a percent rate of how many students earned a passing MT grade, while laying out how many earned a non-passing grade or did not have a grade entered.
- Advisors will reference the intervention tables. There are two - one that summarizes the number of interventions needed, and another that shows student details. The student details sheet will be very short - since I stripped most of the student details from this file, it will appear very bare bones and not that useful. In the final version, it will include information like name, phone and email to enable advisors to perform their outreach.

The inspiration of these tables is from my predecessor who used to make these every semester. He did them all in Excel - they look good and have been useful, but can take a few days to generate. These tables will be remade in plotly - I did some digging and found that there is a graph object specifically for tables (go.Table - https://plotly.com/python/table/) that would allow me to make these tables be filterable using the college and academic period columns I made previously. This improves upon the prior iteration in a few ways:
- A quicker turnaround - using this method would allow for quick creation of the tables for more prompt review and intervention.
- Allow for data reference point - while I am envisioning figures that use this historical data to show changes over time, it would create a single reference point to view how students did in a single term.

With all that said - I will need to do some additional data cleaning first. While I was researching the go.Table function, I realized I would need to do some calculations/summarizations first before I could jump in and make the tables.

In [608]:
#Importing Libraries
import pandas as pd #importing pandas
import numpy as np #importing numpy
import plotly.graph_objects as go #importing the graph objects

**Creating the Summarized DataFrame**

Reflecting on how my predecessor did things, the summary was a view of how *every* enrolled student did in the classes. As such, we'll be using the **mtfullclean.csv** file here. This will allow us to catch all students. Something I would like to explore is making a filter that lets you look at the data for people in the appropriate college (i.e. if you want to view how just CAED students are doing in CAED courses, you have that option). This would help refine the tool so you can understand how majors are doing in the courses.

First things first - we will need to pull in the data!

In [609]:
#Creating the Summarized DataFrame Pt. 1 (Pulling the data in)
mtfull = pd.read_csv("mtfullclean.csv")
mtfull.shape

(85341, 20)

In [610]:
#Creating the Summarized DataFrame Pt. 2 (Checking the Keys)
mtfull.keys()

Index(['Unnamed: 0', 'Record ID', 'Registration Status', 'Subject', 'Course',
       'Campus', 'Final Grade', 'Mid Term Grade', 'Department', 'Major',
       'Academic Period', 'Class', 'Course Code', 'Major College',
       'Subject College', 'Withdrew No MT', 'Dropped no MT', 'Dropped no Fin',
       'Final Grade Number', 'Mid Term Grade Number'],
      dtype='str')

The next thing I want to do is get rid of the drops from the record. Drops do not have a MT grade because they dropped prior to the grade being assigned. This is where the earlier cleaning comes in handy - it lets me identify what records to drop due to being "true" drops.

In [611]:
#Creating the Summarized DataFrame Pt. 3 (Dropping Drops)
mtfullnodrop = mtfull[mtfull["Mid Term Grade"] != "DR"].copy()
mtfullnodrop.shape

(83989, 20)

Next up I want to break down the grade data into three categories - Passing (C or better), Failing (C- or worse) and no Grade Entered (typically due to instructor). 

In [612]:
#Creating the Summarized DataFrame Pt.4 (Checking the values of all the courses)
mtfullnodrop["Course Code"].value_counts()

Course Code
COMM 17591    7821
MUS 23241     4516
MDJ 15027     4181
ARCH 16841    4058
MUS 12716     3398
              ... 
THEA 11844      18
THEA 10123      17
THEA 25303      15
MUS 10925       13
MUS 13040       11
Name: count, Length: 149, dtype: int64

So I wanted to take this a step further, and what I found was that you can combo columns in the () by running it as a list. I learned this from the below site - I figure using course code and academic period is the best way to do this since it'll break it down even further.

https://stackoverflow.com/questions/33271098/get-a-frequency-count-based-on-multiple-dataframe-columns#:~:text=You%20can%20use%20the%20following%20methods%20to,*%20**pivot_table**%20*%20**df_solution%20=%20df.pivot_table(index=%5B'Group'%2C'Size'%5D%2C%20aggfunc='size')**

In [613]:
#Creating the Summarized DataFrame Pt. 5 (breaking out the courses by term)
mtfullnodrop.value_counts(["Course Code","Academic Period"]) 

Course Code  Academic Period
COMM 17591   Fall 2025          1394
             Fall 2024          1388
             Fall 2022          1256
             Fall 2023          1223
ARCH 16841   Fall 2025           691
                                ... 
MUS 13040    Spring 2025           7
MUS 12689    Spring 2023           6
THEA 28661   Spring 2024           5
MUS 10925    Fall 2022             4
MUS 13040    Spring 2023           4
Name: count, Length: 778, dtype: int64

Now I need to make the conditions for columns that highlight when someone is passing/failing at midterms. The criteria is coming from past versions of the report. I'm going to first look at what values there are (so I have a handy cheat sheet), followed by making pass and fail variables.

In [614]:
#Creating the Summarized DataFrame Pt. 6 (Identifying grade values)
mtfullnodrop["Mid Term Grade"].unique()

<StringArray>
[ 'B',  'A',  nan, 'A-',  'C', 'C+',  'D', 'B+', 'B-', 'C-',  'F',  'S', 'D+',
  'W',  'U', 'SF', 'NF']
Length: 17, dtype: str

In [615]:
#Creating the Summarized DataFrame Pt. 7 (Pass/Fail Variables)
mtpass = ["A","A-","B+","B","B-","C+","C","S"]
mtfail = ["C-","D+","D","F","NF","SF","U","W"]

In [616]:
#Creating the Summarized DataFrame Pt. 8 (Making conditions and outputs)
mtstatus = [
    mtfullnodrop["Mid Term Grade"].isin(mtpass),
    mtfullnodrop["Mid Term Grade"].isin(mtfail)
]

mtout = ["MT C or Higher","MT C-, D, F, W"]

In [617]:
#Creating the Summarized DataFrame Pt. 9 (Creating the Status Column)
mtfullnodrop["MT Status"] = np.select(mtstatus, mtout, "MT Not Reported")
mtfullnodrop

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,...,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Status
0,0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,3.7,3.0,MT C or Higher
1,1,1,Registered,AED,22860,KC,A,A,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,4.0,4.0,MT C or Higher
2,2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,...,JR,AED 22860,Other,CAED,False,False,False,4.0,4.0,MT C or Higher
3,3,3,Registered,AED,22860,KC,F,B,ARCH,ID,...,SO,AED 22860,CAED,CAED,False,False,False,0.0,3.0,MT C or Higher
4,4,4,Registered,AED,22860,KC,B,B,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,3.0,3.0,MT C or Higher
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85335,85335,85335,Registered,MDJ,20288,KC,B-,B,MDJ,FM,...,SO,MDJ 20288,CotA,CCI,False,False,False,2.7,3.0,MT C or Higher
85336,85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,...,SO,MDJ 20288,CotA,CCI,False,False,False,3.3,3.0,MT C or Higher
85337,85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,...,JR,MDJ 20288,CotA,CCI,False,False,False,4.0,4.0,MT C or Higher
85339,85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,...,SO,MDJ 20288,CotA,CCI,False,False,False,3.3,3.3,MT C or Higher


The conditions and columns have been made, so now we can make a summary table! To do this, I'm going to use the value counts from earlier, but combo it with the reset index function.

In [618]:
#Creating the Summarized DataFrame Pt. 10 (Making the first table)
mtsummary = mtfullnodrop.value_counts(["Course Code","Academic Period","Subject College","MT Status"]).reset_index()
mtsummary

,Course Code,Academic Period,Subject College,MT Status,count
0,COMM 17591,Fall 2025,CCI,MT C or Higher,1118
1,COMM 17591,Fall 2024,CCI,MT C or Higher,1080
2,COMM 17591,Fall 2022,CCI,MT C or Higher,994
3,COMM 17591,Fall 2023,CCI,MT C or Higher,981
4,COMM 17591,Spring 2026,CCI,MT C or Higher,532
...,...,...,...,...,...
2261,VCD 13182,Spring 2025,CCI,MT Not Reported,1
2262,VCD 17656,Fall 2022,CCI,MT Not Reported,1
2263,VCD 17656,Spring 2025,CCI,"MT C-, D, F, W",1
2264,VCD 17656,Spring 2026,CCI,MT Not Reported,1


So this worked, but I'm going to need a more specific breakdown of how many instances per term - the observation needs to be the count of each type. In it's current state, it isn't that usable. As such, I'm going to use the pivot function (which I found through the pandas cheat sheet)

In [619]:
#Creating the Summarized DataFrame Pt. 11 (Pivot Attempt 1 - Kind of worked, but missing the term)
mtsummary = mtsummary.pivot(columns = "MT Status",
                          values = "count")
mtsummary

MT Status,MT C or Higher,"MT C-, D, F, W",MT Not Reported
0,1118.0,NaN,NaN
1,1080.0,NaN,NaN
2,994.0,NaN,NaN
3,981.0,NaN,NaN
4,532.0,NaN,NaN
...,...,...,...
2261,NaN,NaN,1.0
2262,NaN,NaN,1.0
2263,NaN,1.0,NaN
2264,NaN,NaN,1.0


So rereading the notes about the pivot (and doing some research on the API site), you can use index to specify additional columns - this is where we can bring in the courses, term and college for future filtering. I also opted to fill empty values with a 0. In this case, having blank values makes it look like something was skipped - by specifying 0, we can indicate that no students met that condition.

In [620]:
#Creating the Summarized DataFrame Pt. 12 (Actually working Pivot Table)
mtpivotsummary = mtsummary.pivot(index=["Course Code", "Academic Period","Subject College"],
                            columns = "MT Status",
                            values = "count",).fillna(0)

mtpivotsummary

KeyError: 'Course Code'

In [621]:
#Creating the Summarized DataFrame Pt. 13 (Resetting the INdex to get it to work)
mtpivotsummary = mtpivotsummary.reset_index()
mtpivotsummary

MT Status,level_0,index,Course Code,Academic Period,Subject College,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades
0,0,569,MUS 20682,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
1,1,66,ARCS 12721,Spring 2025,CAED,11.0,0.0,0.0,11.0,1.000000,100.00%
2,2,565,MUS 13040,Spring 2023,CotA,4.0,0.0,0.0,4.0,1.000000,100.00%
3,3,219,DAN 22742,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
4,4,691,THEA 28661,Spring 2024,CotA,5.0,0.0,0.0,5.0,1.000000,100.00%
...,...,...,...,...,...,...,...,...,...,...,...
773,773,606,THEA 11844,Fall 2025,CotA,10.0,5.0,3.0,18.0,0.555556,55.56%
774,774,613,THEA 14029,Spring 2024,CotA,6.0,4.0,1.0,11.0,0.545455,54.55%
775,775,549,MUS 10925,Fall 2022,CotA,2.0,1.0,1.0,4.0,0.500000,50.00%
776,776,608,THEA 13155,Fall 2023,CotA,12.0,7.0,5.0,24.0,0.500000,50.00%


So this worked out! Granted, we now have a random MT Status index column - not ideal, but could be useful if I need to reference some form of Reference ID in the future. I am going to keep it intact for now.

Next though, we need to add additional columns that look at the total number of grades being reported and the percent of how many students are passing.

In [622]:
#Creating the Summarized DataFrame Pt. 14 (Total Grades Column)
mtpivotsummary["Total Grades"] = mtpivotsummary["MT C or Higher"] + mtpivotsummary["MT C-, D, F, W"] + mtpivotsummary["MT Not Reported"]
mtpivotsummary

MT Status,level_0,index,Course Code,Academic Period,Subject College,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades
0,0,569,MUS 20682,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
1,1,66,ARCS 12721,Spring 2025,CAED,11.0,0.0,0.0,11.0,1.000000,100.00%
2,2,565,MUS 13040,Spring 2023,CotA,4.0,0.0,0.0,4.0,1.000000,100.00%
3,3,219,DAN 22742,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
4,4,691,THEA 28661,Spring 2024,CotA,5.0,0.0,0.0,5.0,1.000000,100.00%
...,...,...,...,...,...,...,...,...,...,...,...
773,773,606,THEA 11844,Fall 2025,CotA,10.0,5.0,3.0,18.0,0.555556,55.56%
774,774,613,THEA 14029,Spring 2024,CotA,6.0,4.0,1.0,11.0,0.545455,54.55%
775,775,549,MUS 10925,Fall 2022,CotA,2.0,1.0,1.0,4.0,0.500000,50.00%
776,776,608,THEA 13155,Fall 2023,CotA,12.0,7.0,5.0,24.0,0.500000,50.00%


In [623]:
#Creating the Summarized DataFrame Pt. 15 (# of Passing Grades)
mtpivotsummary["# of Passing Grades"] = mtpivotsummary["MT C or Higher"]/mtpivotsummary["Total Grades"]
mtpivotsummary

MT Status,level_0,index,Course Code,Academic Period,Subject College,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades
0,0,569,MUS 20682,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
1,1,66,ARCS 12721,Spring 2025,CAED,11.0,0.0,0.0,11.0,1.000000,100.00%
2,2,565,MUS 13040,Spring 2023,CotA,4.0,0.0,0.0,4.0,1.000000,100.00%
3,3,219,DAN 22742,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
4,4,691,THEA 28661,Spring 2024,CotA,5.0,0.0,0.0,5.0,1.000000,100.00%
...,...,...,...,...,...,...,...,...,...,...,...
773,773,606,THEA 11844,Fall 2025,CotA,10.0,5.0,3.0,18.0,0.555556,55.56%
774,774,613,THEA 14029,Spring 2024,CotA,6.0,4.0,1.0,11.0,0.545455,54.55%
775,775,549,MUS 10925,Fall 2022,CotA,2.0,1.0,1.0,4.0,0.500000,50.00%
776,776,608,THEA 13155,Fall 2023,CotA,12.0,7.0,5.0,24.0,0.500000,50.00%


The next thing I wanted to do is sort the percent from high to low. Thbis was originally after the conversion to percentage, but I moved it up so it would function properly.

In [624]:
#Creating the Summarized DataFrame Pt. 16 (Sorting from High to Low)
mtpivotsummary = mtpivotsummary.sort_values("# of Passing Grades", ascending = False)
mtpivotsummary

MT Status,level_0,index,Course Code,Academic Period,Subject College,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades
0,0,569,MUS 20682,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
1,1,66,ARCS 12721,Spring 2025,CAED,11.0,0.0,0.0,11.0,1.000000,100.00%
2,2,565,MUS 13040,Spring 2023,CotA,4.0,0.0,0.0,4.0,1.000000,100.00%
3,3,219,DAN 22742,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
4,4,691,THEA 28661,Spring 2024,CotA,5.0,0.0,0.0,5.0,1.000000,100.00%
...,...,...,...,...,...,...,...,...,...,...,...
773,773,606,THEA 11844,Fall 2025,CotA,10.0,5.0,3.0,18.0,0.555556,55.56%
774,774,613,THEA 14029,Spring 2024,CotA,6.0,4.0,1.0,11.0,0.545455,54.55%
775,775,549,MUS 10925,Fall 2022,CotA,2.0,1.0,1.0,4.0,0.500000,50.00%
776,776,608,THEA 13155,Fall 2023,CotA,12.0,7.0,5.0,24.0,0.500000,50.00%


This is where I created a % of Passing Grades column. I struggled to find a way to do this at first - I had to do some research, but then I found a discussion board that discussed this issue and how to resolve it. I did some reading and the format they provided seems to be the best way to do it. You can view the discussion board below:

https://stackoverflow.com/questions/45989858/format-decimals-as-percentages-in-a-column

In [625]:
#Creating the Summarized DataFrame Pt. 17 (Creating a % Column)
mtpivotsummary["% of Passing Grades"] = mtpivotsummary["# of Passing Grades"].apply("{:.2%}".format) #This is changing the formatting of the piece - :.2% is saying keep 2 decimal places and the % is saying multiply it by 100 and add a % sign.
mtpivotsummary

MT Status,level_0,index,Course Code,Academic Period,Subject College,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades
0,0,569,MUS 20682,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
1,1,66,ARCS 12721,Spring 2025,CAED,11.0,0.0,0.0,11.0,1.000000,100.00%
2,2,565,MUS 13040,Spring 2023,CotA,4.0,0.0,0.0,4.0,1.000000,100.00%
3,3,219,DAN 22742,Fall 2024,CotA,9.0,0.0,0.0,9.0,1.000000,100.00%
4,4,691,THEA 28661,Spring 2024,CotA,5.0,0.0,0.0,5.0,1.000000,100.00%
...,...,...,...,...,...,...,...,...,...,...,...
773,773,606,THEA 11844,Fall 2025,CotA,10.0,5.0,3.0,18.0,0.555556,55.56%
774,774,613,THEA 14029,Spring 2024,CotA,6.0,4.0,1.0,11.0,0.545455,54.55%
775,775,549,MUS 10925,Fall 2022,CotA,2.0,1.0,1.0,4.0,0.500000,50.00%
776,776,608,THEA 13155,Fall 2023,CotA,12.0,7.0,5.0,24.0,0.500000,50.00%


**Making the Summary Table in Graph Objects**

Now that the dataframe is all cleaned, I am going to make the actual figure. For this, I am going to use go.Table. This is a graph object that lives in go.Figure. There is a great page on making them on the plotly API site - https://plotly.com/python/table/ - which has coding examples that I was able to reverse engineer. It was really handy to have as a reference while editing and making tables! 

The first figure I made was just making sure I understood the logic, and then I get into the actual table I want to make.

In [626]:
#Making Summary Table in Graph Objects Pt. 1 (Testing the Flow)
fig1 = go.Figure(data=[go.Table( #Running through it as a dictionary
    header = dict(values=["Course Code","Academic Period"]), #Whatever you plug in here determines the headers - DOES NOT NEED TO BE IN THE DATAFRAME
    cells = dict(values=[mtpivotsummary["Course Code"], mtpivotsummary["Academic Period"]]))]) #Where you call the data for the cells - this has to be in the dataframe

fig1.show()

OK - now that I have the flow down, I'm going to make the skeleton of what the table should look like. This is before we make it pretty, so it's going to be a bit rough!

In [627]:
#Making Summary Table in Graph Objects Pt. 2 (Plugging in the skeleton of the final table)
fig2 = go.Figure(data = [go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[mtpivotsummary["Course Code"], 
                        mtpivotsummary["Total Grades"], 
                        mtpivotsummary["MT C-, D, F, W"], 
                        mtpivotsummary["MT Not Reported"],
                        mtpivotsummary["MT C or Higher"],
                        mtpivotsummary["% of Passing Grades"]])
)])

fig2.show()

Now that the table is made, I realized I need to do traces for each college. I used a lot of the code that was from the plotly 2 group - this was especially helpful since there were 4 groups that I am looking at!

In [628]:
#Making Summary Table in Graph Objects Pt. 3 (Making Traces)

alltrace = go.Figure(data = [go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[mtpivotsummary["Course Code"], 
                        mtpivotsummary["Total Grades"], 
                        mtpivotsummary["MT C-, D, F, W"], 
                        mtpivotsummary["MT Not Reported"],
                        mtpivotsummary["MT C or Higher"],
                        mtpivotsummary["% of Passing Grades"]])
)])

caedtrace = go.Figure(data = [go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["Course Code"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["Total Grades"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["MT C-, D, F, W"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["MT Not Reported"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["MT C or Higher"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["% of Passing Grades"]])
)])

ccitrace = go.Figure(data = [go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["Course Code"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["Total Grades"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["MT C-, D, F, W"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["MT Not Reported"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["MT C or Higher"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["% of Passing Grades"]])
)])

cotatrace = go.Figure(data = [go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["Course Code"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["Total Grades"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["MT C-, D, F, W"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["MT Not Reported"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["MT C or Higher"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["% of Passing Grades"]])
)])

layout = go.Layout(
    title=dict(text="Summary of Midterm Grades by Term")
)

fig3 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig3.update_layout(updatemenus = [dict(
    type = "buttons",
    direction = "left",
    showactive = True,
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)])

fig3.show()

ValueError: 
    Invalid element(s) received for the 'data' property of 
        Invalid elements include: [Figure({
    'data': [{'cells': {'values': [['MUS 20682', 'ARCS 12721', 'MUS 13040', 'DAN
                                   22742', 'THEA 28661', 'MDJ 11788', 'THEA 14873',
                                   'MDJ 11788', 'VCD 20802', 'VCD 13182', 'MDJ
                                   12062', 'VCD 17656', 'THEA 28661', 'DAN 13945',
                                   'DAN 22742', 'ARCS 11014', 'MUS 20682', 'ARCS
                                   11014', 'MUS 20682', 'ARTS 23886', 'MDJ 25865',
                                   'DAN 22742', 'MDJ 12062', 'EMAT 12670', 'MDJ
                                   12063', 'EMAT 12670', 'ARCH 22860', 'ARCS
                                   26039', 'MUS 12689', 'MDJ 17554', 'EMAT 10164',
                                   'FDM 17764', 'FDM 20907', 'VCD 18839', 'ARCS
                                   12721', 'MDJ 11788', 'THEA 28661', 'A

So I got ahead of myself - I need to specify go.Table, not go.Figure. I'm basically mushing a bunch of figures into one, so I have to specify the go I want to use here.

In [629]:
#Making Summary Table in Graph Objects Pt. 4 (Making the actual graph objects)

alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[mtpivotsummary["Course Code"], 
                        mtpivotsummary["Total Grades"], 
                        mtpivotsummary["MT C-, D, F, W"], 
                        mtpivotsummary["MT Not Reported"],
                        mtpivotsummary["MT C or Higher"],
                        mtpivotsummary["% of Passing Grades"]])
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["Course Code"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["Total Grades"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["MT C-, D, F, W"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["MT Not Reported"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["MT C or Higher"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["% of Passing Grades"]])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["Course Code"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["Total Grades"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["MT C-, D, F, W"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["MT Not Reported"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["MT C or Higher"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["% of Passing Grades"]])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["Course Code"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["Total Grades"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["MT C-, D, F, W"], 
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["MT Not Reported"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["MT C or Higher"],
                        mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["% of Passing Grades"]])
)

layout = go.Layout(
    title=dict(text="Summary of Midterm Grades by Term")
)

fig3 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig3.update_layout(updatemenus = [dict(
    type = "buttons",
    direction = "left",
    showactive = True,
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)])

fig3.show()

So at this point, I realized that I would need to specify for every term if I wanted to show all of the possible combinations. I did some looking online and found that loops could be a good fix, but I got to thinking even more about the purpose of this table and how it was historically presented. The purpose of this table is to show how students are doing in the current term - not how they are doing in past terms. If someone wanted to do this, they'd have to keep clicking back and forth between current and past terms to find the differences. This would be tedious (and not very user friendly), so I began to think about future iterations of this data to show that. 

After much thought, I decided that I would make a version that would allow me to only show one term, but filterable by college. To this end, I'm going to try and declare a current term variable that, by changing it in one spot, completely updates the file so it can catch just that term when reviewing the data.

In [630]:
#Making Summary Table in Graph Objects Pt. 5 (Single-Term Future Proofing Attempt 1)
currentterm = ["Spring 2026"] #So this is a bit of future proofing - my thought is instead of constantly listing the current term, I can just change the value here when this needs updated.

alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[((mtpivotsummary["Course Code"]) & (mtpivotsummary["Academic Period"] == currentterm)), 
                        ((mtpivotsummary["Total Grades"]) & (mtpivotsummary["Academic Period"] == currentterm)), 
                        ((mtpivotsummary["MT C-, D, F, W"]) & (mtpivotsummary["Academic Period"] == currentterm)), 
                        ((mtpivotsummary["MT Not Reported"]) & (mtpivotsummary["Academic Period"] == currentterm)),
                        ((mtpivotsummary["MT C or Higher"])  & (mtpivotsummary["Academic Period"] == currentterm)),
                        ((mtpivotsummary["% of Passing Grades"]  & (mtpivotsummary["Academic Period"] == currentterm)))])
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[((mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["Course Code"])  & (mtpivotsummary["Academic Period"]  == currentterm)), 
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["Total Grades"])  & (mtpivotsummary["Academic Period"] == currentterm)), 
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["MT C-, D, F, W"]) & (mtpivotsummary["Academic Period"] == currentterm)), 
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["MT Not Reported"]) & (mtpivotsummary["Academic Period"] == currentterm)),
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["MT C or Higher"]) & (mtpivotsummary["Academic Period"] == currentterm)),
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CAED"]["% of Passing Grades"]) & (mtpivotsummary["Academic Period"] == currentterm))])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[((mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["Course Code"])  & (mtpivotsummary["Academic Period"]  == currentterm)), 
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["Total Grades"])  & (mtpivotsummary["Academic Period"] == currentterm)), 
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["MT C-, D, F, W"]) & (mtpivotsummary["Academic Period"] == currentterm)), 
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["MT Not Reported"]) & (mtpivotsummary["Academic Period"] == currentterm)),
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["MT C or Higher"]) & (mtpivotsummary["Academic Period"] == currentterm)),
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CCI"]["% of Passing Grades"]) & (mtpivotsummary["Academic Period"] == currentterm))])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[((mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["Course Code"])  & (mtpivotsummary["Academic Period"]  == currentterm)), 
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["Total Grades"])  & (mtpivotsummary["Academic Period"] == currentterm)), 
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["MT C-, D, F, W"]) & (mtpivotsummary["Academic Period"] == currentterm)), 
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["MT Not Reported"]) & (mtpivotsummary["Academic Period"] == currentterm)),
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["MT C or Higher"]) & (mtpivotsummary["Academic Period"] == currentterm)),
                        ((mtpivotsummary[mtpivotsummary["Subject College"] == "CotA"]["% of Passing Grades"]) & (mtpivotsummary["Academic Period"] == currentterm))])
)

layout = go.Layout(
    title=dict(text="Summary of Midterm Grades by Term")
)

fig4 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig4.update_layout(updatemenus = [dict(
    type = "buttons",
    direction = "left",
    showactive = True,
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)])

fig4.show()

ValueError: ('Lengths must match to compare', (778,), (1,))

Realized at this point where the logic was wrong. Gotta put it all in that frontend where you specify term and college. However, I'm going to instead make variables that I'll call "checks" that ask the data to run and confirm those conditions and only pull the data that passes those conditions. 

In [631]:
#Making Summary Table in Graph Objects Pt. 6 (Future-Proofing Attempt 2)
allcheck = mtpivotsummary["Academic Period"] == "Spring 2026"
caedcheck = (mtpivotsummary["Subject College"] == "CAED") & (mtpivotsummary["Academic Period"] == "Spring 2026")
ccicheck= (mtpivotsummary["Subject College"] == "CCI") & (mtpivotsummary["Academic Period"] == "Spring 2026")
cotacheck= (mtpivotsummary["Subject College"] == "CotA") & (mtpivotsummary["Academic Period"] == "Spring 2026")
currentterm = "Spring 2026"

In [632]:
#Making Summary Table in Graph Objects Pt. 7 (Attempting to make a trace)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[allcheck["Course Code"],
                         allcheck["Total Grades"],
                         allcheck["MT C-, D, F, A"],
                         allcheck["MT Not Reported"],
                         allcheck["MT C or Higher"],
                         allcheck["% of Passing Grades"]])
)

KeyError: 'Course Code'

OK, so we're confused but have the spirit. The issue is that Course Code doesn't have a column in the check (it's trying to read it as a data frame). So what I need to do is make a dataframe using that condition. Since the conditions are written, I just have to make a bunch of data frames that check those conditions now!

In [633]:
#Making Summary Table in Graph Objects Pt. 8 (Making a bunch of smaller dataframes)
allsummary = mtpivotsummary[allcheck]
caedsummary = mtpivotsummary[caedcheck]
ccisummary = mtpivotsummary[ccicheck]
cotasummary = mtpivotsummary[cotacheck]

In [634]:
#Making Summary Table in Graph Objects Pt. 9 (Checking the shape to see if it worked)
caedsummary.shape

(21, 11)

It worked! So now I can pull those dataframes while working. Since they all share the same keys, I'm just going to run a keys trace so I have a quick reference point.

In [635]:
#Making Summary Table in Graph Objects Pt. 10 (Checking the keys for a reference)
caedsummary.keys()

Index(['level_0', 'index', 'Course Code', 'Academic Period', 'Subject College',
       'MT C or Higher', 'MT C-, D, F, W', 'MT Not Reported', 'Total Grades',
       '# of Passing Grades', '% of Passing Grades'],
      dtype='str', name='MT Status')

In [636]:
#Making Summary Table in Graph Objects Pt. 11 (Making a bunch of traces and the figure)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]])
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[caedsummary["Course Code"], 
                        caedsummary["Total Grades"], 
                        caedsummary["MT C-, D, F, W"], 
                        caedsummary["MT Not Reported"], 
                        caedsummary["MT C or Higher"], 
                        caedsummary["% of Passing Grades"]])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[ccisummary["Course Code"], 
                        ccisummary["Total Grades"], 
                        ccisummary["MT C-, D, F, W"], 
                        ccisummary["MT Not Reported"], 
                        ccisummary["MT C or Higher"], 
                        ccisummary["% of Passing Grades"]])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]),
    cells = dict(values=[cotasummary["Course Code"], 
                        cotasummary["Total Grades"], 
                        cotasummary["MT C-, D, F, W"], 
                        cotasummary["MT Not Reported"], 
                        cotasummary["MT C or Higher"], 
                        cotasummary["% of Passing Grades"]])
)

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades")
)

fig5 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig5.update_layout(updatemenus = [dict(
    type = "buttons",
    direction = "left",
    showactive = True,
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)])

fig5.show()

That is satisying. But now the next steps are to pretty it up! I'm going to be referencing the plotly API's discussion of go.Table - they have a lot of examples that I could reverse engineer for this visualization. For the button customization, I'm gonna recall the plotly groups' lectures and pull information for making the buttons. For the sake of my sanity, I am going to make the figure be just the initial table - that way I can toy with it and don't have a lot of scrolling to do

In [637]:
#Making Summary Table in Graph Objects Pt. 12 (Making it pretty)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "black", #I'm thinking that making the lines black will not only make them more distinct, but help with differentiating the headers
                fill_color = "#003976", #I'm going to pull the school's primary color for this - it may not mesh well with the black lines though
                font_color = "white", #this should pop well against the blue background
                font_weight = "bold"  #to help it stand out further, going to embolden the font
                ), 
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                       line_color = "black", #keeping in line with the header, going to try black lines
                       fill_color = [["#FFFFFF", "#CCE5FF"] * 50] #So this one I did do a bit of looking for to see how to achieve it. 
                       # If you make a list of colors and multiply it, it makes the colors loop. A good tip I found was make the number big - that way it will for sure cover it all!
                ))

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades")
)

fig6 = go.Figure(data = [alltrace], layout = layout)

fig6.show()

So I like the colors of it all, but don't like the header lines. Plus, not a big fan of how "small" the table is. It isn't clear that it is scrollable, so I'm going to add a touch more height to it.

In [638]:
#Making Summary Table in Graph Objects Pt. 13 (Making it More Pretty)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray", #I need something that pops more, but not white. I found this color while looking at examples and it may work
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"
                ), 
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                       line_color = "black",
                       fill_color = [["#FFFFFF", "#CCE5FF"] * 50]
                ))

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades"),
    height = 650
)

fig7 = go.Figure(data = [alltrace], layout = layout)

fig7.show()

That's much nicer! Now I'm going to apply it to all the traces, plus re-add the buttons and customize it plus the title.

In [639]:
#Making Summary Table in Graph Objects Pt. 14 (Adding the buttons and title customization)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[caedsummary["Course Code"], 
                        caedsummary["Total Grades"], 
                        caedsummary["MT C-, D, F, W"], 
                        caedsummary["MT Not Reported"], 
                        caedsummary["MT C or Higher"], 
                        caedsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[ccisummary["Course Code"], 
                        ccisummary["Total Grades"], 
                        ccisummary["MT C-, D, F, W"], 
                        ccisummary["MT Not Reported"], 
                        ccisummary["MT C or Higher"], 
                        ccisummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[cotasummary["Course Code"], 
                        cotasummary["Total Grades"], 
                        cotasummary["MT C-, D, F, W"], 
                        cotasummary["MT Not Reported"], 
                        cotasummary["MT C or Higher"], 
                        cotasummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades",
                font_weight = "bold", #I wanted to make the title more defined, so I made it bold
                xanchor = "center", #same idea here - I want to make it centered on the page
                yanchor = "top", #I figure from the top would be better since I could control how far down I want the title to be
                x = .5, #I believe this should be in the middle of the page
                y = .9), #Same idea here - I think this should be a little from the top, but could be wrong.
    height = 650
)

fig8 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig8.update_layout(updatemenus = [dict(
    type = "buttons",
    direction = "left",
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = .8, #So tried following the same logic as the title - I want the buttons right underneath it
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)])

fig8.show()

So it worked, just not well. I'm going to tinker with the y values to make them higher up. I'll do a lot of little tinkering, so below will be the final tinkering (# of attempts to get it right = 17).

Something else I'd like to do is add a title to the menu, as well as make it a drop down menu. 

In [640]:
#Making Summary Table in Graph Objects Pt. 15 (Tinkering with the locations)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[caedsummary["Course Code"], 
                        caedsummary["Total Grades"], 
                        caedsummary["MT C-, D, F, W"], 
                        caedsummary["MT Not Reported"], 
                        caedsummary["MT C or Higher"], 
                        caedsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[ccisummary["Course Code"], 
                        ccisummary["Total Grades"], 
                        ccisummary["MT C-, D, F, W"], 
                        ccisummary["MT Not Reported"], 
                        ccisummary["MT C or Higher"], 
                        ccisummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[cotasummary["Course Code"], 
                        cotasummary["Total Grades"], 
                        cotasummary["MT C-, D, F, W"], 
                        cotasummary["MT Not Reported"], 
                        cotasummary["MT C or Higher"], 
                        cotasummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades",
                font_weight = "bold", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650
)

fig9 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig9.update_layout(updatemenus = [dict(
    type = "dropdown", #this will transform it into a dropdown menu
    direction = "down", #I want it to go down
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1, #So tried following the same logic as the title - I want the buttons right underneath it
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)],
    annotations=[dict( #This is adding the note for the dropdown - I followed the custom buttons example on the plotly site - https://plotly.com/python/custom-buttons/
        text = "College:", 
        showarrow = False,
        xanchor = "center", #setting the same x/y variables for the annotation - I'm lowering the x and y values a bit so they appear to the left of the menu
        yanchor = "top",
        x = .45,
        y = 1.08
    )])

fig9.show()

And that is the base table down. There is an extra element I want to add though, and that is color coding the "% of Passing Grades" column. In the file made in Excel, fixed color values were used for certain ranges:
- 85%+ = Green
- 80 - 84.9% = Yellow
- Sub-80% = Red

I like this idea and want to implement it in my figure. At first, I considered making a gradient instead of setting "hard" colors. However, I worry that this would get thrown off in terms where there are not that many classes or there are a concentration of high pass rates. I think this would skew the colors where high performing classes would show as red and give the wrong impression. Setting the fixed colors would appeal to biological relevance (green = good, yellow = warning, red = bad), so people could get a quick grasp on how many courses are "problem" courses by just quickly glancing and see the amount of red/yellow/green.

After considering the best way to implement this, I think a loop would be best. It could loop through the criteria until it settles on a color (check green, then yellow, then red). I'm going to keep the same guidelines as the old version of the file (as listed above). I think I'll need to make a unique loop for each summarized table, but I may be able to get by with one since it is in each? Worst case I'll have to make multiple.

In [641]:
#Making Summary Table in Graph Objects Pt. 16 (Making Pass/Fail Colors - Attempt 1)
allpass = []

for num in allsummary["% of Passing Grades"]:
    if num >= 85:
        allpass.append("#CCFFCC") #So this is a newer thing - I did some looking and it is recommended to use .append() when you want to change factors based on a loop - this is saying if it meets the condition, use this
    elif num >= 80: #I thought this would be else if, but I double checked the logic and found that elif is used here, but in Java you write else if
        allpass.append("#FFFFCC")
    else:
        allpass.append("#FFCCCC")

TypeError: '>=' not supported between instances of 'str' and 'int'

So I am getting an error indicating that I cannot use this since one is a string and the other is an integer. I'm guessing it is because of the % sign. I thought I could go back and reference an earlier table pre-%, and that worked!

In [642]:
#Making Summary Table in Graph Objects Pt. 17 (Making Pass/Fail Colors - Attempt 2)
allpass = []

for num in allsummary["# of Passing Grades"]:
    if num >= .85: #I had to add the decimal - since this is pre-%, it is in the decimal format.
        allpass.append("#CCFFCC")
    elif num >= .80:
        allpass.append("#FFFFCC")
    else:
        allpass.append("#FFCCCC")

It worked that time! Now let's see if we can get this to work (going to just edit the alltrace like always - a bit easier that way!)

In [643]:
#Making Summary Table in Graph Objects Pt. 18 (Adding the Pass/Fail Colors - Attempt 1)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#FFFFFF", "#FFFFFF", "#FFFFFF", "#FFFFFF", allpass], ["#CCE5FF", "#CCE5FF", "#CCE5FF", "#CCE5FF", "#CCE5FF", allpass]]) #I'm attempting to replicate the zebra striping by indicating all the individual values and ending with the color
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[caedsummary["Course Code"], 
                        caedsummary["Total Grades"], 
                        caedsummary["MT C-, D, F, W"], 
                        caedsummary["MT Not Reported"], 
                        caedsummary["MT C or Higher"], 
                        caedsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[ccisummary["Course Code"], 
                        ccisummary["Total Grades"], 
                        ccisummary["MT C-, D, F, W"], 
                        ccisummary["MT Not Reported"], 
                        ccisummary["MT C or Higher"], 
                        ccisummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[cotasummary["Course Code"], 
                        cotasummary["Total Grades"], 
                        cotasummary["MT C-, D, F, W"], 
                        cotasummary["MT Not Reported"], 
                        cotasummary["MT C or Higher"], 
                        cotasummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades",
                font_weight = "bold", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650
)

fig10 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig10.update_layout(updatemenus = [dict(
    type = "dropdown", #this will transform it into a dropdown menu
    direction = "down", #I want it to go down
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1, #So tried following the same logic as the title - I want the buttons right underneath it
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)],
    annotations=[dict( #This is adding the note for the dropdown - I followed the custom buttons example on the plotly site - https://plotly.com/python/custom-buttons/
        text = "College:", 
        showarrow = False,
        xanchor = "center", #setting the same x/y variables for the annotation - I'm lowering the x and y values a bit so they appear to the left of the menu
        yanchor = "top",
        x = .45,
        y = 1.08
    )])

fig10.show()

That didn't work - it ended up spitting out a bunch of black rows, which is the opposite of what I need. Let me try doing just one row - I can specify colors for each column.

In [644]:
#Making Summary Table in Graph Objects Pt. 19 (Making the Pass/Fail Colors - Attempt 2)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = ["#FFFFFF", "#FFFFFF", "#FFFFFF", "#FFFFFF", "#FFFFFF", allpass])
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[caedsummary["Course Code"], 
                        caedsummary["Total Grades"], 
                        caedsummary["MT C-, D, F, W"], 
                        caedsummary["MT Not Reported"], 
                        caedsummary["MT C or Higher"], 
                        caedsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[ccisummary["Course Code"], 
                        ccisummary["Total Grades"], 
                        ccisummary["MT C-, D, F, W"], 
                        ccisummary["MT Not Reported"], 
                        ccisummary["MT C or Higher"], 
                        ccisummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[cotasummary["Course Code"], 
                        cotasummary["Total Grades"], 
                        cotasummary["MT C-, D, F, W"], 
                        cotasummary["MT Not Reported"], 
                        cotasummary["MT C or Higher"], 
                        cotasummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades",
                font_weight = "bold", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650
)

fig11 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig11.update_layout(updatemenus = [dict(
    type = "dropdown", #this will transform it into a dropdown menu
    direction = "down", #I want it to go down
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1, #So tried following the same logic as the title - I want the buttons right underneath it
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)],
    annotations=[dict( #This is adding the note for the dropdown - I followed the custom buttons example on the plotly site - https://plotly.com/python/custom-buttons/
        text = "College:", 
        showarrow = False,
        xanchor = "center", #setting the same x/y variables for the annotation - I'm lowering the x and y values a bit so they appear to the left of the menu
        yanchor = "top",
        x = .45,
        y = 1.08
    )])

fig11.show()

So it worked, but I have to sacrifice the alternating row colors. I did turn to Gemini at this point to see what I can find. The prompt I gave it was:

*I would like to have an alternating color pattern for a go.Table, but the last row should use a color that changes based on the value. How can I do this?*

 It's first attempt was introducing some crazy code to loop the alternating colors. While a nice idea, I don't want to introduce something too complex. So I asked it to step away from that "doing WITHOUT" the complex code. It recommended just specifying each column for each row, kind of like how I did earlier. I'm going to try that now.

In [645]:
#Making Summary Table in Graph Objects Pt. 20 (Making it work)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50, #This is what it recommended - breaking up the columns individually, so the first one is Row 1, second is Row 2, followed by the *50 to repeat
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            allpass])
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[caedsummary["Course Code"], 
                        caedsummary["Total Grades"], 
                        caedsummary["MT C-, D, F, W"], 
                        caedsummary["MT Not Reported"], 
                        caedsummary["MT C or Higher"], 
                        caedsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[ccisummary["Course Code"], 
                        ccisummary["Total Grades"], 
                        ccisummary["MT C-, D, F, W"], 
                        ccisummary["MT Not Reported"], 
                        ccisummary["MT C or Higher"], 
                        ccisummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[cotasummary["Course Code"], 
                        cotasummary["Total Grades"], 
                        cotasummary["MT C-, D, F, W"], 
                        cotasummary["MT Not Reported"], 
                        cotasummary["MT C or Higher"], 
                        cotasummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [["#FFFFFF", "#CCE5FF"] * 50])
)

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades",
                font_weight = "bold", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650
)

fig12 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig12.update_layout(updatemenus = [dict(
    type = "dropdown", #this will transform it into a dropdown menu
    direction = "down", #I want it to go down
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1, #So tried following the same logic as the title - I want the buttons right underneath it
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)],
    annotations=[dict( #This is adding the note for the dropdown - I followed the custom buttons example on the plotly site - https://plotly.com/python/custom-buttons/
        text = "College:", 
        showarrow = False,
        xanchor = "center", #setting the same x/y variables for the annotation - I'm lowering the x and y values a bit so they appear to the left of the menu
        yanchor = "top",
        x = .45,
        y = 1.08
    )])

fig12.show()

So I had to do some tweaking, but I go it to work! It didn't include the *50, so I had to add that so it'd loop. Adding it to each column color is annoying, but it cleaned up nicely! Now to add it in to the other traces

In [646]:
#Making Summary Table in Graph Objects Pt. 21 (Plugging it all in)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            allpass])
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[caedsummary["Course Code"], 
                        caedsummary["Total Grades"], 
                        caedsummary["MT C-, D, F, W"], 
                        caedsummary["MT Not Reported"], 
                        caedsummary["MT C or Higher"], 
                        caedsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            allpass])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[ccisummary["Course Code"], 
                        ccisummary["Total Grades"], 
                        ccisummary["MT C-, D, F, W"], 
                        ccisummary["MT Not Reported"], 
                        ccisummary["MT C or Higher"], 
                        ccisummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            allpass])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[cotasummary["Course Code"], 
                        cotasummary["Total Grades"], 
                        cotasummary["MT C-, D, F, W"], 
                        cotasummary["MT Not Reported"], 
                        cotasummary["MT C or Higher"], 
                        cotasummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            allpass])
)

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades",
                font_weight = "bold", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650
)

fig13 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig13.update_layout(updatemenus = [dict(
    type = "dropdown", #this will transform it into a dropdown menu
    direction = "down", #I want it to go down
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1, #So tried following the same logic as the title - I want the buttons right underneath it
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)],
    annotations=[dict( #This is adding the note for the dropdown - I followed the custom buttons example on the plotly site - https://plotly.com/python/custom-buttons/
        text = "College:", 
        showarrow = False,
        xanchor = "center", #setting the same x/y variables for the annotation - I'm lowering the x and y values a bit so they appear to the left of the menu
        yanchor = "top",
        x = .45,
        y = 1.08
    )])

fig13.show()

So it carried over the pass rate color to everyone, but not in the way I wanted. I'll have to specify each summary table to have their own color. Below will be that, followed by the updated table logic.

In [647]:
#Making Summary Table in Graph Objects Pt. 22 (Everyone's Pass Rate)
caedpass = []

for num in caedsummary["# of Passing Grades"]:
    if num >= .85: #I had to add the decimal - since this is pre-%, it is in the decimal format.
        caedpass.append("#CCFFCC")
    elif num >= .80:
        caedpass.append("#FFFFCC")
    else:
        caedpass.append("#FFCCCC")

ccipass = []

for num in ccisummary["# of Passing Grades"]:
    if num >= .85: #I had to add the decimal - since this is pre-%, it is in the decimal format.
        ccipass.append("#CCFFCC")
    elif num >= .80:
        ccipass.append("#FFFFCC")
    else:
        ccipass.append("#FFCCCC")

cotapass = []

for num in cotasummary["# of Passing Grades"]:
    if num >= .85: #I had to add the decimal - since this is pre-%, it is in the decimal format.
        cotapass.append("#CCFFCC")
    elif num >= .80:
        cotapass.append("#FFFFCC")
    else:
        cotapass.append("#FFCCCC")

In [648]:
#Making Summary Table in Graph Objects Pt. 23 (Plugging in all the colors)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            allpass])
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[caedsummary["Course Code"], 
                        caedsummary["Total Grades"], 
                        caedsummary["MT C-, D, F, W"], 
                        caedsummary["MT Not Reported"], 
                        caedsummary["MT C or Higher"], 
                        caedsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            caedpass])
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[ccisummary["Course Code"], 
                        ccisummary["Total Grades"], 
                        ccisummary["MT C-, D, F, W"], 
                        ccisummary["MT Not Reported"], 
                        ccisummary["MT C or Higher"], 
                        ccisummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ccipass])
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[cotasummary["Course Code"], 
                        cotasummary["Total Grades"], 
                        cotasummary["MT C-, D, F, W"], 
                        cotasummary["MT Not Reported"], 
                        cotasummary["MT C or Higher"], 
                        cotasummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            cotapass])
)

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades",
                font_weight = "bold", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650
)

fig14 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig14.update_layout(updatemenus = [dict(
    type = "dropdown", #this will transform it into a dropdown menu
    direction = "down", #I want it to go down
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1, #So tried following the same logic as the title - I want the buttons right underneath it
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)],
    annotations=[dict( #This is adding the note for the dropdown - I followed the custom buttons example on the plotly site - https://plotly.com/python/custom-buttons/
        text = "College:", 
        showarrow = False,
        xanchor = "center", #setting the same x/y variables for the annotation - I'm lowering the x and y values a bit so they appear to the left of the menu
        yanchor = "top",
        x = .45,
        y = 1.08
    )])

fig14.show()

So I got it to work, but something I just realized while looking at the visualization again is that it is showing the CotA version of the table by default. I'm unsure as to why this is - I looked back at the plotly group's notes and there did not seem to be an issue without specifying it. I'm going to add on a bit to each trace that states "visible" true for the alltrace and false for the others to see if that sorts it out.

In [649]:
#Making Summary Table in Graph Objects Pt. 24 (Hiding Everyone)
alltrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[allsummary["Course Code"], 
                        allsummary["Total Grades"], 
                        allsummary["MT C-, D, F, W"], 
                        allsummary["MT Not Reported"], 
                        allsummary["MT C or Higher"], 
                        allsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            allpass]),
    visible = True #By specifying true here and false elsewhere, this will show this table only when it loads. Without it, it'll show the "newest" table first (which is CotA)
)

caedtrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[caedsummary["Course Code"], 
                        caedsummary["Total Grades"], 
                        caedsummary["MT C-, D, F, W"], 
                        caedsummary["MT Not Reported"], 
                        caedsummary["MT C or Higher"], 
                        caedsummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            caedpass]),
    visible = False
)

ccitrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[ccisummary["Course Code"], 
                        ccisummary["Total Grades"], 
                        ccisummary["MT C-, D, F, W"], 
                        ccisummary["MT Not Reported"], 
                        ccisummary["MT C or Higher"], 
                        ccisummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ccipass]),
    visible = False
)

cotatrace = go.Table(
    header = dict(values = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values=[cotasummary["Course Code"], 
                        cotasummary["Total Grades"], 
                        cotasummary["MT C-, D, F, W"], 
                        cotasummary["MT Not Reported"], 
                        cotasummary["MT C or Higher"], 
                        cotasummary["% of Passing Grades"]],
                        line_color = "black",
                        fill_color = [
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            ['#FFFFFF', '#CCE5FF'] * 50,
                            cotapass]),
    visible = False
)

layout = go.Layout(
    title=dict(text= "Summary of Spring 2026 Midterm Grades",
                font_weight = "bold", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650
)

fig15 = go.Figure(data = [alltrace, caedtrace, ccitrace, cotatrace], layout = layout)

fig15.update_layout(updatemenus = [dict(
    type = "dropdown", #this will transform it into a dropdown menu
    direction = "down", #I want it to go down
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1, #So tried following the same logic as the title - I want the buttons right underneath it
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)],
    annotations=[dict( #This is adding the note for the dropdown - I followed the custom buttons example on the plotly site - https://plotly.com/python/custom-buttons/
        text = "College:", 
        showarrow = False,
        xanchor = "center", #setting the same x/y variables for the annotation - I'm lowering the x and y values a bit so they appear to the left of the menu
        yanchor = "top",
        x = .45,
        y = 1.08
    )])

fig15.show()

With that made, I think I can put the summary table to rest. It shows a single term, lets you filter by college, and color codes the % of passing grades. All in all, I think it looks pretty snazzy!

Something I did for fun is consult Google Gemini on how would I add an additional filter. While it is nice to filter by college, I figure the next step would be by subject so you could see what the pass rates for just that subject. They indicated that it is possible, but you need to use Dash for hosting since it wouldn't be able to pick up multiple filters if it is an HTML file.  Apparently, when you try to do multiple filters as an HTML file, it won't register both as active. So if I changed Filter A and then Filter B, it would only show what Filter B, not both. This is a concerning limitation for the future of my project, especially when I get to making additional visualizations (Heat Maps, Line Charts, etc.). 

*This will likely be something I ask about in class/may make an appointment for office hours to go over that*

**Advisor Intervention Rates**

So the next table I'm making is going to make is looking at a summary of the number of interventions needed in a semester. This is going to result in a shift from the **mtfullclean.csv** file and into the **mthubinterventions.csv** file. I'm also going to use the same logic like I did for the **Summary of Spring 2026 Midterm Grades** above - this is a file that is looked at in a single term, so it will focus on the single term. Also just like before, we will have to do some cleaning first and then move into making the table. I'm going to be reusing a lot of code, so there won't be as many comments as before.

In [650]:
#Advisor Intervention Rates Table Pt. 1 (Pulling in the data)
mtinterfull = pd.read_csv("mthubinterventions.csv")
mtinterfull.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,...,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Grade Status
0,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,NaN,2.0,FR/SO Intervention Needed
1,14,14,Registered,AED,22860,KC,C,D,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,2.0,1.0,FR/SO Intervention Needed
2,19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,...,SO,AED 22860,CAED,CAED,False,False,False,2.7,2.0,FR/SO Intervention Needed
3,31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,...,SO,AED 22860,CAED,CAED,False,False,False,0.0,1.0,FR/SO Intervention Needed
4,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,2.7,1.7,FR/SO Intervention Needed


In [651]:
#Advisor Intervention Rates Table Pt. 2 (Checking The Keys)
mtinterfull.keys()

Index(['Unnamed: 0', 'Record ID', 'Registration Status', 'Subject', 'Course',
       'Campus', 'Final Grade', 'Mid Term Grade', 'Department', 'Major',
       'Academic Period', 'Class', 'Course Code', 'Major College',
       'Subject College', 'Withdrew No MT', 'Dropped no MT', 'Dropped no Fin',
       'Final Grade Number', 'Mid Term Grade Number', 'MT Grade Status'],
      dtype='str')

In [652]:
#Advisor Intervention Rates Table Pt. 3 (Confirming the Values)
mtinterfull["Registration Status"].unique()

<StringArray>
[                'Registered', 'Stopped Attending - Failed',
    'Never Attended - Failed',              'Admin Dropped',
              'Std Withdrawn',            'Admin Withdrawn']
Length: 6, dtype: str

Now that we've got the file pulled and have confirmation about what is shown in the file, we can start working on the data to make a table that shows the # of interventions needed this semester. I'm going to approach this differently. Since we can use .count(), I can quickly make the table. Plus, it doesn't need that much prettying up, which makes it nicer too!

In [653]:
#Advisor Intervention Rates Table Pt. 4 (Checking the Counts)
mtinterfull["MT Grade Status"].value_counts()

MT Grade Status
FR/SO Intervention Needed    5460
JR/SR Intervention Needed     652
Name: count, dtype: int64

In [654]:
#Advisor Intervention Rates Table Pt. 5 (Failed Pivot - Has duplicate values)
intercountpivot = mtinterfull.pivot(index=["Academic Period", "Major College"],
                            columns = "MT Grade Status",
                            values = "MT Grade Status")

intercountpivot

ValueError: Index contains duplicate entries, cannot reshape

In [655]:
#Advisor Intervention Rates Table Pt. 6 (Failed Attempt - Can't group by MT Grade Status)
intercountpivot = mtinterfull.pivot_table(index=["Academic Period", "Major College"],
                            columns = "MT Grade Status",
                            values = "MT Grade Status",
                            aggfunc = "count") #I thought this might be the issue - it is 

intercountpivot

ValueError: Grouper for 'MT Grade Status' not 1-dimensional

At this point I did some looking to see what I could find for why it isn't working. In this case, I need to use the pivot_table() function to get it to work.

In [656]:
#Advisor Intervention Rates Table Pt. 7 (Working Pivot Table)
intercountpivot = mtinterfull.pivot_table(index=["Academic Period", "Major College"],
                            columns = "MT Grade Status",
                            values = "Record ID",
                            aggfunc = "count")

intercountpivot

MT Grade Status                FR/SO Intervention Needed  \
Academic Period Major College                              
Fall 2022       CAED                                 133   
                CCI                                  209   
                CotA                                 424   
Fall 2023       CAED                                 157   
                CCI                                  210   
                CotA                                 442   
Fall 2024       CAED                                 165   
                CCI                                  181   
                CotA                                 406   
Fall 2025       CAED                                 147   
                CCI                                  180   
                CotA                                 417   
Spring 2023     CAED                                 122   
                CCI                                  155   
                CotA                                 392   
Spring 2024     CAED                                 115   
                CCI                                  145   
                CotA                                 404   
Spring 2025     CAED                                 142   
                CCI                                  127   
                CotA                                 363   
Spring 2026     CAED                                  84   
                CCI                                   72   
                CotA                                 268   

MT Grade Status                JR/SR Intervention Needed  
Academic Period Major College                             
Fall 2022       CAED                                  15  
                CCI                                   49  
                CotA                                  37  
Fall 2023       CAED                                   6  
                CCI                                   26  
                CotA                                  30  
Fall 2024       CAED                                  11  
                CCI                                   35  
                CotA                                  51  
Fall 2025       CAED                                   9  
                CCI                                   22  
                CotA                                  24  
Spring 2023     CAED                                  18  
                CCI                                   32  
                CotA                                  35  
Spring 2024     CAED                                  28  
                CCI                                   28  
                CotA                                  38  
Spring 2025     CAED                                  17  
                CCI                                   22  
                CotA                                  46  
Spring 2026     CAED                                  14  
                CCI                                   19  
                CotA                                  40

In [657]:
#Advisor Intervention Rates Table Pt. 8 (Making it work out)
intercountpivot = intercountpivot.reset_index()
intercountpivot

MT Grade Status,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed
0,Fall 2022,CAED,133,15
1,Fall 2022,CCI,209,49
2,Fall 2022,CotA,424,37
3,Fall 2023,CAED,157,6
4,Fall 2023,CCI,210,26
5,Fall 2023,CotA,442,30
6,Fall 2024,CAED,165,11
7,Fall 2024,CCI,181,35
8,Fall 2024,CotA,406,51
9,Fall 2025,CAED,147,9


In [658]:
#Advisor Intervention Rates Table Pt. 9 (Adding a total column)
intercountpivot["Total # of Instances"] = intercountpivot["FR/SO Intervention Needed"] + intercountpivot["JR/SR Intervention Needed"]
intercountpivot

MT Grade Status,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed,Total # of Instances
0,Fall 2022,CAED,133,15,148
1,Fall 2022,CCI,209,49,258
2,Fall 2022,CotA,424,37,461
3,Fall 2023,CAED,157,6,163
4,Fall 2023,CCI,210,26,236
5,Fall 2023,CotA,442,30,472
6,Fall 2024,CAED,165,11,176
7,Fall 2024,CCI,181,35,216
8,Fall 2024,CotA,406,51,457
9,Fall 2025,CAED,147,9,156


In [659]:
#Advisor Intervention Rates Table Pt. 10 (Creating the logic for each college)
allintercheck = intercountpivot["Academic Period"] == "Spring 2026"
caedintercheck = (intercountpivot["Major College"] == "CAED") & (intercountpivot["Academic Period"] == "Spring 2026")
cciintercheck = (intercountpivot["Major College"] == "CCI") & (intercountpivot["Academic Period"] == "Spring 2026")
cotaintercheck = (intercountpivot["Major College"] == "CotA") & (intercountpivot["Academic Period"] == "Spring 2026")
interterm = "Spring 2026"

In [660]:
#Advisor Intervention Rates Table Pt. 11 (Making dataframes)
allinter = intercountpivot[allintercheck]
caedinter = intercountpivot[caedintercheck]
cciinter = intercountpivot[cciintercheck]
cotainter = intercountpivot[cotaintercheck]

Now that we've got our frames, I can get to work on the figure! I'm going to pull in an older table just to have as a quick reference when making the new ones

In [ ]:
#Advisor Intervention Rates Table Pt. 12 (Pulling in an old table to have as an example)
#fig1 = go.Figure(data=[go.Table(
#    header = dict(values=["Course Code","Academic Period"]),
#    cells = dict(values=[mtpivotsummary["Course Code"], mtpivotsummary["Academic Period"]]))])

#fig1.show()

In [661]:
#Advisor Intervention Rates Table Pt.  13 (The first pass)
interfig1 = go.Figure(data = [go.Table(
    header=dict(values = ["Group #", "Hub Midterm Intervention - Student Criteria", "# of Instances"]),
    cells = dict(values = [
                [1,2," "],
                 ["Hub Major, FR or SO, Hub Course, midterm D+ or Below, still registered", 
                  "Hub Major, JR or SR, Hub Course, midterm F, still registered",
                  "Total"],
                 [allinter["FR/SO Intervention Needed"], allinter["JR/SR Intervention Needed"], allinter["Total # of Instances"]]])
)])

interfig1.show()

So that worked, but it didn't at the same time. It pulled the data in, just as a list and not the actual sum. I'm going to need to make a sum for each row so that way it works.

In [662]:
#Advisor Intervention Rates Table Pt. 14 (Making Sum Totals)
frsoallinstances = sum(allinter["FR/SO Intervention Needed"])
jrsrallinstances = sum(allinter["JR/SR Intervention Needed"])
totalallinstances = sum(allinter["Total # of Instances"])

In [663]:
#Advisor Intervention Rates Table Pt. 15 (Running it back to see if it worked)
interfig2 = go.Figure(data = [go.Table(
    header=dict(values = ["Group #", "Hub Midterm Intervention - Student Criteria", "# of Instances"]),
    cells = dict(values = [
                [1,2," "],
                 ["Hub Major, FR or SO, Hub Course, midterm D+ or Below, still registered", 
                  "Hub Major, JR or SR, Hub Course, midterm F, still registered",
                  "Total"],
                 [frsoallinstances, jrsrallinstances, totalallinstances]]))])

interfig2.show()

There we go - now it works! Time to run it back and do all the traces.... again.... yay....

In [664]:
#Advisor Intervention Rates Table Pt. 16 (Making the traces)
allintertrace = go.Table(
    header=dict(values = ["Group #", "Hub Midterm Intervention - Student Criteria", "# of Instances"]),
    cells = dict(values = [
                [1,2," "], #Since these are not unique values, I can specify the numbers here. I'm leaving hte last one blank since there isn't a "Total" group
                 ["Hub Major, FR or SO, Hub Course, midterm D+ or Below, still registered", #same as above, but with text. I'll tweak these a bit between tables
                  "Hub Major, JR or SR, Hub Course, midterm F, still registered",
                  "Total"],
                 [frsoallinstances, jrsrallinstances, totalallinstances]]),
    visible = True)

caedintertrace = go.Table(
    header=dict(values = ["Group #", "CAED Midterm Intervention - Student Criteria", "# of Instances"]),
    cells = dict(values = [
                [1,2," "],
                 ["CAED Major, FR or SO, CAED Course, midterm D+ or Below, still registered", 
                  "CAED Major, JR or SR, CAED Course, midterm F, still registered",
                  "Total"],
                 [caedinter["FR/SO Intervention Needed"], caedinter["JR/SR Intervention Needed"], caedinter["Total # of Instances"]]]),
    visible = False)

cciintertrace = go.Table(
    header=dict(values = ["Group #", "CCI Midterm Intervention - Student Criteria", "# of Instances"]),
    cells = dict(values = [
                [1,2," "],
                 ["CCI Major, FR or SO, CCI Course, midterm D+ or Below, still registered", 
                  "CCI Major, JR or SR, CCI Course, midterm F, still registered",
                  "Total"],
                 [cciinter["FR/SO Intervention Needed"], cciinter["JR/SR Intervention Needed"], cciinter["Total # of Instances"]]]),
    visible = False)

cotaintertrace = go.Table(
    header=dict(values = ["Group #", "CotA Midterm Intervention - Student Criteria", "# of Instances"]),
    cells = dict(values = [
                [1,2," "],
                 ["CotA Major, FR or SO, CotA Course, midterm D+ or Below, still registered", 
                  "CotA Major, JR or SR, CotA Course, midterm F, still registered",
                  "Total"],
                 [cotainter["FR/SO Intervention Needed"], cotainter["JR/SR Intervention Needed"], cotainter["Total # of Instances"]]]),
    visible = False)

layout = go.Layout(
    title=dict(text="Number of Required Advisor Interventions")
)

interfig3 = go.Figure(data = [allintertrace, caedintertrace, cciintertrace, cotaintertrace], layout = layout)

interfig3.update_layout(updatemenus = [dict(
    type = "buttons",
    direction = "left",
    showactive = True,
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)])

interfig3.show()


Now we have the buttons, all the traces, and the title. Now I can go in and do the "prettying up" of the file. For the sake of consistency, I'm going to use the same design system as last time. I will be making some smaller tweaks, and I'll highlight those where appropriate.

In [665]:
#Advisor Intervention Rates Table Pt. 17 (Making it Pretty)
allintertrace = go.Table(
    header=dict(values = ["Group #", "Hub Midterm Intervention - Student Criteria", "# of Instances"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["Hub Major, FR or SO, Hub Course, midterm D+ or Below, still registered", 
                  "Hub Major, JR or SR, Hub Course, midterm F, still registered",
                  "Total"],
                 [frsoallinstances, jrsrallinstances, totalallinstances]],
                 line_color = [["black", "black", "#003976"], ["black", "black", "#003976"], ["black", "black", "#003976"]], #So since the last column is the total, I needed to change the properties to lists (similar to above)
                 fill_color = [["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"]],
                 font_color = [["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"]]),
    visible = True)

caedintertrace = go.Table(
    header=dict(values = ["Group #", "CAED Midterm Intervention - Student Criteria", "# of Instances"],                
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["CAED Major, FR or SO, CAED Course, midterm D+ or Below, still registered", 
                  "CAED Major, JR or SR, CAED Course, midterm F, still registered",
                  "Total"],
                 [caedinter["FR/SO Intervention Needed"], caedinter["JR/SR Intervention Needed"], caedinter["Total # of Instances"]]],
                 line_color = [["black", "black", "#003976"], ["black", "black", "#003976"], ["black", "black", "#003976"]],
                 fill_color = [["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"]],
                 font_color = [["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"]]),
    visible = False)

cciintertrace = go.Table(
    header=dict(values = ["Group #", "CCI Midterm Intervention - Student Criteria", "# of Instances"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["CCI Major, FR or SO, CCI Course, midterm D+ or Below, still registered", 
                  "CCI Major, JR or SR, CCI Course, midterm F, still registered",
                  "Total"],
                 [cciinter["FR/SO Intervention Needed"], cciinter["JR/SR Intervention Needed"], cciinter["Total # of Instances"]]],
                 line_color = [["black", "black", "#003976"], ["black", "black", "#003976"], ["black", "black", "#003976"]],
                 fill_color = [["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"]],
                 font_color = [["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"]]),
    visible = False)

cotaintertrace = go.Table(
    header=dict(values = ["Group #", "CotA Midterm Intervention - Student Criteria", "# of Instances"],
                line_color = "lightslategray",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["CotA Major, FR or SO, CotA Course, midterm D+ or Below, still registered", 
                  "CotA Major, JR or SR, CotA Course, midterm F, still registered",
                  "Total"],
                 [cotainter["FR/SO Intervention Needed"], cotainter["JR/SR Intervention Needed"], cotainter["Total # of Instances"]]],
                 line_color = [["black", "black", "#003976"], ["black", "black", "#003976"], ["black", "black", "#003976"]],
                 fill_color = [["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"]],
                 font_color = [["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"]]),
    visible = False)

layout = go.Layout(
    title=dict(text="Number of Required Advisor Interventions",
            font_weight = "bold", 
            xanchor = "center", 
            yanchor = "top", 
            x = .5,
             y = .965),
    height = 650
)

interfig3 = go.Figure(data = [allintertrace, caedintertrace, cciintertrace, cotaintertrace], layout = layout)

interfig3.update_layout(updatemenus = [dict(
    type = "dropdown", 
    direction = "down", 
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1,
    buttons = 
        list([dict(label = "All",
            method = "update",
            args = [{"visible":[True,False,False,False]}]),
        dict(label = "CAED",
            method = "update",
            args = [{"visible":[False,True,False,False]}]),
        dict(label = "CCI",
            method = "update",
            args = [{"visible":[False,False,True,False]}]),
        dict(label = "CotA",
            method = "update",
         args = [{"visible":[False,False,False,True]}])])
)],
    annotations=[dict( 
        text = "College:", 
        showarrow = False,
        xanchor = "center",
        yanchor = "top",
        x = .45,
        y = 1.08
    )])

interfig3.show()


I've got the table-making piece down pat - really it is the masking that I'm struggling with. I think I just need to do it more and it'll come more natural. Either way - the (small) table is done, so we can move onto my final milestone for the week!

**Creating a Contact List for Interventions**

Due to anonymization of the data, this is probably going to be the least helpful table of the three. The purpose of this table is less informative, serving as the contact list for advisors so they know who to contact. However, since we have no names or contact information, this list would not be very helpful to advisors! We have all the core parts needed to make the list - what we are really missing are things like the phone number and email. I contemplated adding in fake data for those cells - but there is a lot of variety in that data that would be hard to capture (namely course title, name, and email). For reference though, here are the columns that *should* be there, but cannot due to the nature of the project:
- Section #
- CRN
- Course Title
- Chosen Name
- Email
- Phone #
- ID (We will sub in Record ID here)

These will be added in the final release.

Let's begin!

In [666]:
#Creating a Contact List Pt. 1 (Pulling the Data)
intercontact = pd.read_csv("mthubinterventions.csv")
intercontact.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,...,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Grade Status
0,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,NaN,2.0,FR/SO Intervention Needed
1,14,14,Registered,AED,22860,KC,C,D,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,2.0,1.0,FR/SO Intervention Needed
2,19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,...,SO,AED 22860,CAED,CAED,False,False,False,2.7,2.0,FR/SO Intervention Needed
3,31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,...,SO,AED 22860,CAED,CAED,False,False,False,0.0,1.0,FR/SO Intervention Needed
4,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,2.7,1.7,FR/SO Intervention Needed


Next we need to add contact groups. In this current iteration, there are just 2 (Fr/So and Jr/Sr). As we add classes with additional special needs (i.e. courses that require a higher grade), more groups will be added. The thought is that this will be presented alongside the table above, so it will be a reference for what the numbers mean.

In [667]:
#Creating a Contact List Pt. 2 (Making the Contact Groups)
intercontact["MT Intervention Group"] = np.where(intercontact["MT Grade Status"] == "FR/SO Intervention Needed", 1, 2)
intercontact.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,...,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Grade Status,MT Intervention Group
0,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,...,AED 22860,CAED,CAED,False,False,False,NaN,2.0,FR/SO Intervention Needed,1
1,14,14,Registered,AED,22860,KC,C,D,ARCH,ID,...,AED 22860,CAED,CAED,False,False,False,2.0,1.0,FR/SO Intervention Needed,1
2,19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,...,AED 22860,CAED,CAED,False,False,False,2.7,2.0,FR/SO Intervention Needed,1
3,31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,...,AED 22860,CAED,CAED,False,False,False,0.0,1.0,FR/SO Intervention Needed,1
4,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,...,AED 22860,CAED,CAED,False,False,False,2.7,1.7,FR/SO Intervention Needed,1


Now we need to add the cells that advisors will use to enter information about their meeting with the students - like notes, pin, holds and who contacted them. They will be left purposely empty so they can be filled out with blank data.

In [668]:
#Creating a Contact List Pt. 3 (Adding the Advisor entry cells)
intercontact[["Notes", "Advising Pin", "Hold", "Advisor", "Date"]] = np.nan
intercontact.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,...,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Grade Status,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date
0,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,...,False,NaN,2.0,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
1,14,14,Registered,AED,22860,KC,C,D,ARCH,ID,...,False,2.0,1.0,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
2,19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,...,False,2.7,2.0,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
3,31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,...,False,0.0,1.0,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
4,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,...,False,2.7,1.7,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN


Now that we have everything we need (well, close to everything), we can drop columns we do not need and reorder those that are left behind!

In [669]:
#Creating a Contact List Pt. 4 (Creating a Key Reference)
intercontact.keys()

Index(['Unnamed: 0', 'Record ID', 'Registration Status', 'Subject', 'Course',
       'Campus', 'Final Grade', 'Mid Term Grade', 'Department', 'Major',
       'Academic Period', 'Class', 'Course Code', 'Major College',
       'Subject College', 'Withdrew No MT', 'Dropped no MT', 'Dropped no Fin',
       'Final Grade Number', 'Mid Term Grade Number', 'MT Grade Status',
       'MT Intervention Group', 'Notes', 'Advising Pin', 'Hold', 'Advisor',
       'Date'],
      dtype='str')

In [670]:
#Creating a Contact List Pt. 5 (Dropping Unneeded columns)
intercontactdrop = intercontact.drop(columns = ["Unnamed: 0", 'Final Grade', "Subject College", "Withdrew No MT", 'Dropped no MT', 'Dropped no Fin', 'Final Grade Number', 'Mid Term Grade Number', 'MT Grade Status', 'Registration Status','Campus'], axis=0)
intercontactdrop

,Record ID,Subject,Course,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date
0,11,AED,22860,C,ARCH,ID,Fall 2022,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
1,14,AED,22860,D,ARCH,ID,Fall 2022,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
2,19,AED,22860,C,ARCH,ID,Fall 2022,SO,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
3,31,AED,22860,D,ARCH,ID,Fall 2022,SO,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
4,33,AED,22860,C-,ARCH,ID,Fall 2022,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6107,85275,MDJ,20288,F,MDJ,FM,Fall 2022,SR,MDJ 20288,CotA,2,NaN,NaN,NaN,NaN,NaN
6108,85309,MDJ,20288,F,MDJ,DMP,Spring 2023,JR,MDJ 20288,CCI,2,NaN,NaN,NaN,NaN,NaN
6109,85315,MDJ,20288,D,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,1,NaN,NaN,NaN,NaN,NaN
6110,85316,MDJ,20288,C-,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,1,NaN,NaN,NaN,NaN,NaN


I'm also going to change Record ID --> ID. This is how it looks on the original report, so this step will be cut later once that report is made.

In [671]:
#Creating a Contact List Pt. 6 (Changing Record ID --> ID)
intercontactdrop = intercontactdrop.rename(columns = {"Record ID":"ID"})
intercontactdrop

,ID,Subject,Course,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date
0,11,AED,22860,C,ARCH,ID,Fall 2022,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
1,14,AED,22860,D,ARCH,ID,Fall 2022,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
2,19,AED,22860,C,ARCH,ID,Fall 2022,SO,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
3,31,AED,22860,D,ARCH,ID,Fall 2022,SO,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
4,33,AED,22860,C-,ARCH,ID,Fall 2022,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6107,85275,MDJ,20288,F,MDJ,FM,Fall 2022,SR,MDJ 20288,CotA,2,NaN,NaN,NaN,NaN,NaN
6108,85309,MDJ,20288,F,MDJ,DMP,Spring 2023,JR,MDJ 20288,CCI,2,NaN,NaN,NaN,NaN,NaN
6109,85315,MDJ,20288,D,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,1,NaN,NaN,NaN,NaN,NaN
6110,85316,MDJ,20288,C-,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,1,NaN,NaN,NaN,NaN,NaN


Now that we have the cleaned up table, we can reorder the columns so they reflect appropriately. I will note that this is something I'm hoping to consult with the advising team on at a later date. That way I can make sure the table is containing the information they really need.

In [672]:
#Creating a Contact List Pt. 7 (Cleaned up Key Reference)
intercontactdrop.keys()

Index(['ID', 'Subject', 'Course', 'Mid Term Grade', 'Department', 'Major',
       'Academic Period', 'Class', 'Course Code', 'Major College',
       'MT Intervention Group', 'Notes', 'Advising Pin', 'Hold', 'Advisor',
       'Date'],
      dtype='str')

In [673]:
#Creating a Contact List Pt. 8 (Reordering the columns)
intercontactorder = intercontactdrop[['MT Intervention Group','Notes','Advising Pin','Hold','Advisor', 'Date','Major','Class','Mid Term Grade','Department','Subject','Course','ID','Academic Period','Major College']]
intercontactorder.head()

,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date,Major,Class,Mid Term Grade,Department,Subject,Course,ID,Academic Period,Major College
0,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C,ARCH,AED,22860,11,Fall 2022,CAED
1,1,NaN,NaN,NaN,NaN,NaN,ID,FR,D,ARCH,AED,22860,14,Fall 2022,CAED
2,1,NaN,NaN,NaN,NaN,NaN,ID,SO,C,ARCH,AED,22860,19,Fall 2022,CAED
3,1,NaN,NaN,NaN,NaN,NaN,ID,SO,D,ARCH,AED,22860,31,Fall 2022,CAED
4,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C-,ARCH,AED,22860,33,Fall 2022,CAED


In [674]:
#Creating a Contact List Pt. 9 (Sorting the groups)
intercontactorder = intercontactorder.sort_values(by="MT Intervention Group")
intercontactorder

,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date,Major,Class,Mid Term Grade,Department,Subject,Course,ID,Academic Period,Major College
0,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C,ARCH,AED,22860,11,Fall 2022,CAED
1,1,NaN,NaN,NaN,NaN,NaN,ID,FR,D,ARCH,AED,22860,14,Fall 2022,CAED
2,1,NaN,NaN,NaN,NaN,NaN,ID,SO,C,ARCH,AED,22860,19,Fall 2022,CAED
3,1,NaN,NaN,NaN,NaN,NaN,ID,SO,D,ARCH,AED,22860,31,Fall 2022,CAED
4,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C-,ARCH,AED,22860,33,Fall 2022,CAED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6107,2,NaN,NaN,NaN,NaN,NaN,FM,SR,F,MDJ,MDJ,20288,85275,Fall 2022,CotA
6108,2,NaN,NaN,NaN,NaN,NaN,DMP,JR,F,MDJ,MDJ,20288,85309,Spring 2023,CCI
3641,2,NaN,NaN,NaN,NaN,NaN,FM,SR,SF,FDM,FDM,20907,49404,Spring 2025,CotA
3693,2,NaN,NaN,NaN,NaN,NaN,FD,SR,F,FDM,FDM,27868,49928,Spring 2024,CotA


OK - so we now have the setup for the list finished. Like I mentioned before, I want to signify where the missing columns will go:
- Section #, CRN and Course Title = After Course
- Chosen Name, Email, Phone #  = After ID

The logic behind this flow is to keep the course information together, followed by student contact information.

**What Am I Doing With the Data?**

So I've come to a crossroads now. I have the table ready, but I need to decide how to host it. I was considering making it a go.Table and hosting it online, but that would defeat it's purpose (it isn't editable or downloadable). In the same vein, I don't want to give a raw csv file to the advisors to complete - it doesn't look nice, plus if they try to create new tabs they won't save (something I learned the hard way before). 

I opted to consult Google Gemini for some ideas. I explained the nature of the project (a class project that is planned to be rolled out as a viable tool) and what I would like to do with it. We ran through a lot of options, and I wanted to go through what was provided:
- It first brought up just moving it to PowerBI - while I do think *some* of these may end up in PowerBI, I indicated that is not what I intend/want to happen with this piece specifically - it needs to be downloaded and editable.
- It brought up Streamlit as an option - Streamlit is a program that lets you build data apps quickly. This sounded appealling - you could host the app online, they could edit it on the fly, etc. But there were a few issues with this. First, it has a free option, but I worry that I'd hit a paywall sooner or later - I want to be as low cost as possible. Second, I'd need a server that I could host the app on. I unfortunately do not have a server that I could do this (albeit the public one students get - which would not be a good idea). Third, based on what I read, I'd need a machine running 24/7 to run the app. For something that only needs ran twice a year, this seemed impractical. For all these reasons, Streamlist was shot down as an option (for now - if I get deep into coding, I could consider this as an option).
- It then brought up just using the .styler function in pandas, styling the data that way, but when you export to CSV it would lose that formatting. You'd have to do it as an Excel file, which isn't ideal.

I kicked around with Gemini the idea of manually running this process twice a year, downloading the file for me to check out, and then uploading it to Sharepoint and linking it on a "Midterm Grade" Page that advisors could access and edit the file directly. In response to this, it recommended a different program called XlsxWriter- this is a Python library geared toward the creation of Excel XLSX files, which would help with formatting and carrying stuff over to Excel.

Since this is a whole new python library, I did do some reliance on Gemini for writing the initial code and explaining it to me. You can see the code (with my tweaks/comments explaining what is happening below). The prompt I gave it was the following:

*I think the XLSXWriter approach is preferred. Could you help me by writing some basic code for exporting the data to an Excel file with zebra stripes, custom headers, and putting it in a table? My dataframe is called intercontactorder. Additionally, please explain each step so I can understand what is happening for future XLSXWriter projects. Thank you!*

In [675]:
#Making the Excel File Pt. 1 (Learning the Library Pt. 1)
writer = pd.ExcelWriter("202610_MT_Intervention_List.xlsx", #Here we create our object, plus make the workbook that we are dumping the data in (it is empty now) 
                        engine = "xlsxwriter") #This is defining the engine that we are using to edit the file - this is what lets us make the data look nice!

intercontactorder.to_excel(writer, #So this is us transmitting the data over to the Excel file
                           index = False, #This is skipping the index
                           sheet_name = "Intervention") #This is naming the sheet it is going on - and specifying we are dumping it here!

Having done this first step, I realize that I may want to do this separately (one for each college) - that way it doesn't make one long list. However, I am going to stick with making a "master" version first, followed up by a single file with multiple tabs later.

In [676]:
#Making the Excel File Pt. 2 (Learning the Library Pt. 2)
workbook = writer.book #So this is defining the entire workbook. Here we are saying any changes we make to this should affect the entire workbook (not just a specific sheet)
worksheet = writer.sheets['Intervention'] #Same as above, just to the specific sheet now

In [677]:
#Making the Excel File Pt. 3 (Learning the Library Pt. 3)
(max_row, max_col) = intercontactorder.shape #So this is defining the maximum dimensions of the Excel file - using shape makes it as big as the shape file

column_settings = [{'header': column} for column in intercontactorder.columns] #This is making the header names based on the dataframe

In [678]:
#Making the Excel File Pt. 4 (Learning the Library Pt. 4)
worksheet.add_table(0, 0, max_row, max_col - 1, { #Now we are making the table - this is setting the dimensions of the table (0,0 is A1, while the rest is from shape. The -1 makes sure we catch all the columns)
    "columns": column_settings, #calling out what we want to go in the column headers
    "style": "Table Style Medium 9" #Calling out the style - this is the same as the styling used in Excel, so I just have to find the name there when I am doing my designing!
})

0

In [679]:
#Making the Excel File Pt. 5 (Learning the Library Pt. 5)
worksheet.set_column(0, max_col-1, 20) #this is controlling the column width - saying cover the whole thing, and make each column 20 units wide

worksheet.freeze_panes(1,1) #This is freezing the pane so when you scroll, you see that top row

writer.close() #this is just saving the file

I love this library. This is going to make not just this project easier, but every project I work on in Python easier. I just have to learn how to speak it first. 

So now that we are done with the "experimentation" phase of the project, I am going to work on making versions of the dataframe for each college - since each college's advisors will contact students separately, this will let me make a sheet for each unit!

In [680]:
#Making the Excel File Pt. 6 (Attempting to Use the Old Table and failing)
#I thought I could be sneaky and use the cold checks to make this table - nope, couldn't do it :(
caedcontacts = intercontactorder[caedintercheck]
ccicontacts = intercontactorder[cciintercheck]
cotacontacts = intercontactorder[cotaintercheck]

C:\Users\thilber2\AppData\Local\Temp\ipykernel_12744\4059141630.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  caedcontacts = intercontactorder[caedintercheck]


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [681]:
#Making the Excel File Pt. 7 (Making the new variables)
caedintercontactcheck = (intercontactorder["Major College"] == "CAED") & (intercontactorder["Academic Period"] == "Spring 2026")
cciintercontactcheck = (intercontactorder["Major College"] == "CCI") & (intercontactorder["Academic Period"] == "Spring 2026")
cotaintercontactcheck = (intercontactorder["Major College"] == "CotA") & (intercontactorder["Academic Period"] == "Spring 2026")

In [682]:
#Making the Excel File Pt. 8 (making the new dataframes)
caedcontacts = intercontactorder[caedintercontactcheck]
ccicontacts = intercontactorder[cciintercontactcheck]
cotacontacts = intercontactorder[cotaintercontactcheck]

Now that we have the criteria written out, I'm going to make the sheets. I figure that, since this is for a single hub, we could provide a single Excel file to them and advisors could work on their tab (where appropriate). 

In [683]:
#Making the Excel File Pt. 9 (Making the Excel File)
interwriter = pd.ExcelWriter("Spring_2026_Midterm_Intervention_List.xlsx",
                             engine = "xlsxwriter")

caedcontacts.to_excel(interwriter, index = False, sheet_name = "CAEDInterventions")
ccicontacts.to_excel(interwriter, index = False, sheet_name = "CCIInterventions")
cotacontacts.to_excel(interwriter, index = False, sheet_name = "CotAInterventions")

In [684]:
#Making the Excel File Pt. 10 (Creating the Workbook and Sheets Failed Attempt)
interbook = interwriter.book
intersheets = interwriter.sheets["CAEDInterventions", "CCIInterventions","CotAInterventions"] #Turns out you can only make a variable for a single sheet - can't do them all in one :(

KeyError: ('CAEDInterventions', 'CCIInterventions', 'CotAInterventions')

In [685]:
#Making the Excel File Pt. 11 (Everyone gets a variable)
interbook = interwriter.book
caedintersheet = interwriter.sheets["CAEDInterventions"]
cciintersheet = interwriter.sheets["CCIInterventions"]
cotaintersheet = interwriter.sheets["CotAInterventions"]

Now that we have out workbook and sheets made, I want to explore customizing the look. Based on the library's documentation (https://xlsxwriter.readthedocs.io/format.html) you need to specify the format separate from everything else, and can ultimately toss it in where appropriate. Since that "column settings" variable Gemini recommended is editable and can be plugged in elsewhere, I'm going to make some variables for custom formatting to get that going!

In [686]:
#Making the Excel File Pt. 12 (Making the wrong formatting)
header_format = interbook.add_format({#this function lets you specify the formatting you want done to the notebook
    "font_name": "Andale WT", #This is the font that is standard in Cognos
    "font_size": 8, #This is the standard font size
    "bold": True, #This is to embolden the header
    "font_color":"#FFFFFF", #Using this, the white will pop on hte blue backgrounf
    "fill_color": "#003976",
     "align":"center" #Keeping with the style guide - but the wrong color function
})

AttributeError: 'Format' object has no attribute 'set_fill_color'

In [687]:
#Making the Excel File Pt. 13 (Making the correct formatting)
header_format = interbook.add_format({
    "font_name": "Andale WT",
    "font_size": 8,
    "bold": True,
    "font_color":"#FFFFFF",
    "bg_color": "#003976",
    "align":"center"
})

#Applying the formatting to each sheet
caedcolumns = [{'header': column, "header_format":header_format} for column in caedcontacts.columns] #Making the column settings for each sheet
ccicolumns = [{'header': column, "header_format":header_format} for column in ccicontacts.columns] #Making the column settings for each sheet
cotacolumns = [{'header': column, "header_format":header_format} for column in cotacontacts.columns] #Making the column settings for each sheet

Something else that I wanted to do was customize the rows as well. I want to have the same font as the above, but I want to have the alternating rows. Reading up on it, the logic is different from that of go.Table(can't indicate every other row), plus I want to have a selection of columns be highlighted red. This is one I caved again and consulted Gemini on. What I asked was the following:

*How can I set the logic up so the rows have alternating colors and select columns are highlighted red. The impacted columns are columns G, H and I*

In [688]:
#Making the Excel File Pt. 14 (Setting up "Zebra Rows")
zebraformat1 = interbook.add_format({"bg_color":"#FFFFFF", #so this is defining the format of the rows - follows the same logic throughout (font style, size, and color)
                                    "font_name": "Andale WT",
                                    "font_size": 8,
                                    "align": "center"})

zebraformat2= interbook.add_format({"bg_color":'#CCE5FF',
                                    "font_name": "Andale WT",
                                    "font_size": 8,
                                    "align": "center"})

alertformat = interbook.add_format({"bg_color":'#FFCCCC',
                                    "font_name": "Andale WT",
                                    "font_size": 8,
                                    "align": "center"
})

So below is a series of attempts to make the spreadsheet look how I wanted. If you try to run them all, Python starts acting up since you cannot override a pre-existing table. As such, I commented out all of them except the ones that work (the last few). This helped me avoid accidentally running them and having to start over (which happened more than I care to admit). The first comment indicates what was wrong with the attempt. 

The logic of going through it was "Try a slightly different thing on each, run it, and see which took." If none took, start over with new blocks.

In [ ]:
#CAED's Worksheet Customization - Attempt 1 (Didn't work - only colored headers)
#(caedrowmax, caedcolmax) = caedcontacts.shape

#caedintersheet.add_table(0,0,caedrowmax, caedcolmax - 1, {
#    "columns": caedcolumns,
#    "style": None
#})

#caedintersheet.conditional_format(1,0,caedrowmax, caedcolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) = 0",
#    "format": zebraformat1
#})

#caedintersheet.conditional_format(1,0,caedrowmax, caedcolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) != 0",
#    "format": zebraformat2
#})

#caedintersheet.conditional_format(1,6,caedrowmax,8,{
#    "type":"no_errors",
#    "format": alertformat
#})

In [ ]:
#CCI's Worksheet Customization (Attempt 1 - Didn't work, only colored headers)
#(ccirowmax, ccicolmax) = ccicontacts.shape

#cciintersheet.add_table(0,0,ccirowmax, ccicolmax - 1, {
#    "columns": ccicolumns,
#    "style": None
#})

#cciintersheet.conditional_format(1,0,ccirowmax, ccicolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) = 0",
#    "format": zebraformat1
#})

#cciintersheet.conditional_format(1,0,ccirowmax, ccicolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) != 0",
#    "format": zebraformat2
#})

#cciintersheet.conditional_format(1,6,ccirowmax,8,{
#    "type":"no_errors",
#    "format": alertformat
#})

In [ ]:
#CCI's Worksheet Customization (Attempt 2 - Didn't work, removed all formatting)
#(ccirowmax, ccicolmax) = ccicontacts.shape

#cciintersheet.add_table(0,0,1, ccicolmax - 1, {
#    "columns": ccicolumns,
#    "style": None
#})

#cciintersheet.conditional_format(1,0,1, ccicolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) = 0",
#    "format": zebraformat1
#})

#cciintersheet.conditional_format(1,0,1, ccicolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) != 0",
#    "format": zebraformat2
#})

#cciintersheet.conditional_format(1,6,1,8,{
#    "type":"no_errors",
#    "format": alertformat
#})

In [ ]:
#CotA's Worksheet Customization (Attempt 1 - Didn't work, only colored headers)
#(cotarowmax, cotacolmax) = cotacontacts.shape

#cotaintersheet.add_table(0,0,cotarowmax, cotacolmax - 1, {
#    "columns": cotacolumns,
#    "style": None
#})

#cotaintersheet.conditional_format(1,0,cotarowmax, cotacolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) = 0",
#    "format": zebraformat1
#})

#cotaintersheet.conditional_format(1,0,cotarowmax, cotacolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) != 0",
#    "format": zebraformat2
#})

#cotaintersheet.conditional_format(1,6,cotarowmax,8,{
#    "type":"no_errors",
#    "format": alertformat
#})

In [ ]:
#CotA's Worksheet Customization (Attempt 2 - removed all formatting)
#(cotarowmax, cotacolmax) = cotacontacts.shape

#cotaintersheet.add_table(0,0,cotarowmax, cotacolmax - 1, {
#    "columns": cotacolumns,
#    "style": "Table Style Light 1"
#})

#cotaintersheet.conditional_format(1,0,1, cotacolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) = 0",
#    "format": zebraformat1
#})

#cotaintersheet.conditional_format(1,0,1, cotacolmax - 1, {
#    "type": "formula",
#    "criteria": "=MOD(ROW(),2) != 0",
#    "format": zebraformat2
#})

#cotaintersheet.conditional_format(1,6,1,8,{
#    "type":"no_errors",
#    "format": alertformat
#})

In [ ]:
#CotA's Worksheet Customization (Attempt 3 - more complicated logic, still no work and removed all formatting)
#(cotarowmax, cotacolmax) = cotacontacts.shape

#cotaintersheet.add_table(0, 0, cotarowmax, cotacolmax - 1, {
#    "columns": cotacolumns,
#    "style": "Table Style Light 1"
#})

#cotaintersheet.conditional_format(1, 6, cotarowmax, 8, {
#    "type": "no_errors",
#    "format": alertformat
#})

#cotaintersheet.conditional_format(1, 0, cotarowmax, cotacolmax - 1, {
#    "type": "formula",
#    "criteria": "=AND(MOD(ROW(),2)=0, COLUMN()<7, COLUMN()>9)", 
#    "format": zebraformat1
#})

#cotaintersheet.conditional_format(1, 0, cotarowmax, cotacolmax - 1, {
#    "type": "formula",
#    "criteria": "=AND(MOD(ROW(),2)!=0, COLUMN()<7, COLUMN()>9)", 
#    "format": zebraformat2
#})

Below are the ones that I finally got to work. I decided to stop fighting with the MOD formula that Gemini provided. It was good in context (apply to ever other row), but it was something I struggled to understand and opted to move away from and use a pre-built style instead. This lets me still use the custom headers, just not custom colors for the rows.

In [689]:
#CCI's Worksheet Customization (Attempt 3 - THIS ONE WORKS)
(ccirowmax, ccicolmax) = ccicontacts.shape #Setting the max/min columns

#all rows formatting
cciintersheet.add_table(0, 0, ccirowmax, ccicolmax - 1, { #Establishing the look for the whole thing
    "columns": ccicolumns, #specifying the column format
    "style": "Table Style Medium 2" # This gives light blue zebra stripes
})

# Formatting for the columns I want to highlight
cciintersheet.conditional_format(1, 6, ccirowmax, 8, {#This limits the impact to just those rows
    "type": "no_errors", #this is saying to look at all the cells in that range - just ignore the broken ones
    "format": alertformat #pull the alert format from earlier
})

0

In [690]:
#CAED's Worksheet Customization (Working ver.)
(caedrowmax, caedcolmax) = caedcontacts.shape

caedintersheet.add_table(0, 0, caedrowmax, caedcolmax - 1, {
    "columns": caedcolumns,
    "style": "Table Style Medium 2" 
})

caedintersheet.conditional_format(1, 6, caedrowmax, 8, {
    "type": "no_errors",
    "format": alertformat
})

0

In [691]:
#CotA's Worksheet Customization (Working ver.)
(cotarowmax, cotacolmax) = cotacontacts.shape

cotaintersheet.add_table(0, 0, cotarowmax, cotacolmax - 1, {
    "columns": cotacolumns,
    "style": "Table Style Medium 2" 
})

cotaintersheet.conditional_format(1, 6, cotarowmax, 8, {
    "type": "no_errors",
    "format": alertformat
})

0

While that all worked, it still didn't create the formatting for the rows I wanted. I still wanted to have the fonts and alignment change, so I had to make a variable that I'd call later.

In [692]:
#Making the Excel File Pt. 15 (Row customization)
rowcustoms = interbook.add_format({
    "font_name": "Andale WT",
    "font_size": 8,
    "align": "center"
})

Now I can go ahead and do the final formatting where the column widths get adjusted. I had two versions - the first pass (which wasn't that great) and then the second pass (which was more percise and allowed for more room in the notes column. This is also where we plug in that rowcustoms variable to change the font.

In [ ]:
#Making the Excel File Pt. 16 (First Excel File Formatting - DON"T RUN THIS ONE)
caedintersheet.set_column(0, caedcolmax-1, 20, rowcustoms)
caedintersheet.freeze_panes(1,0)

cciintersheet.set_column(0, ccicolmax-1, 20, rowcustoms)
cciintersheet.freeze_panes(1,0)

cotaintersheet.set_column(0, cotacolmax-1, 20, rowcustoms)
cotaintersheet.freeze_panes(1,0)

interwriter.close()

In [ ]:
#Making the Excel File Pt. 17 (True Excel File Formatting)
caedintersheet.set_column(0, caedcolmax-1, 20, rowcustoms)
caedintersheet.set_column(1, 1, 80, rowcustoms) #This allows for the notes row to be wide - by calling out just this column, we can ensure advisors have enough space to write notes.
caedintersheet.freeze_panes(1,0) #This freezes the top pane for easy reference while scrolling through the file

cciintersheet.set_column(0, ccicolmax-1, 20, rowcustoms)
cciintersheet.set_column(1, 1, 80, rowcustoms)
cciintersheet.freeze_panes(1,0)

cotaintersheet.set_column(0, cotacolmax-1, 20, rowcustoms)
cotaintersheet.set_column(1, 1, 80, rowcustoms)
cotaintersheet.freeze_panes(1,0)

interwriter.close()

**Milestone 2 Reflection**

I found this week to be very satisfying. Most of the stuff I did this week I've done in Excel before (manipulating data to fit a table, organizing rows, preparing reports, etc.). However, this opened a path for me to automate the report-writing process. I can see many instances where the XLSXWriter library could help me automate reports I regularly run, I would just need to devote time to writing the code so I can quickly create the reports. To this end, I think the midterm grades are a good test run of this - they hit multiple audiences, there's a need to automate this process due to the college growing, and it is easy to get feedback on it (due to the number of people who work on it).

Reflecting on the whole process, I wish I was better at loops. The times that I consulted Gemini, it proposed writing loops instead of manually writing stuff out. I was reading the code it generated and, while I could follow along with it, I couldn't totally grasp what it was doing. I think if I scale this up further (i.e. to cover the whole university, not just the CCI/CotA/CAED hub), using loops would become inevitable. Another roadblock I hit was the limits placed on me by the university. I am unsure if I would be able to secure a server to store and run these programs - at least in my current role. As I move forward and do more work, that is something I could try to persuade the university to sponsor. However, the ability to host/create similar reports on Sharepoint/PowerBI does allow for a suitable replacement (at least for now).

Looking ahead to the next steps, I've spent a lot of time considering what the goal of the final milestone should be. So far, I've been focused on cleaning and toolmaking - important skills to have and valuable deliverables for the college, but I'm not wading into the interactive data side of things as much as I think I should be. Reviewing the data options before me, I see a few visualizations that I could consider making:
- A line chart that shows the pass rate over time for courses. Due to the limitations of Sharepoint & Plotly Graph Objects, I think creating 3 visualizations (one for each college) would work, with a filter to select course(s). I know there is a PowerBI application that you can plug plotly scripts into and it will generate the visualization, so maybe comboing those together to have a toggle for college in PowerBI + the filters from plotly would work? I could also make multiple subplots to show core courses by subject, but I think that's starting to get a bit too intense.
- I'd also like to make a Heat Map that shows the indication of midterm grades as a indicator of final grades. This would try to plot students on a chart to see where indicators of success would be (i.e. students who get an A often get a A on finals). While not a "definitive" indicator since there are so many factors to consider, this could help highlight how advisor interventions are helping. This does present a roadblock though - how do I tackle the filter problem? I think having multiple filters (major, course, college, term) would be valuable and make the data more useful, but it is not compatible with how I'm going to host this.
- A line chart that shows interventions over time. Honestly, this is probably the least intensive one out of the above, as it just shows the rates at which interventions have gone up and down over time - nothing too crazy!

All in all, I think I'm on track to have a successful third milestone submission, I just need to overcome the technical limitations first. 